In [1]:
import os

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
import bs4
import requests
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv, dotenv_values

load_dotenv(override=True)
# env_dict = dotenv_values(".env")
# print("--- Contents of .env file ---")
# for key, value in env_dict.items():
#     # Masking the value so you don't accidentally print your full API key to your screen!
#     masked_value = f"{value[:5]}...{value[-4:]}" if len(value) > 10 else value
#     print(f"{key}: {masked_value}")

/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/ipykernel_77144/3902444640.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


True

In [2]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
docs = loader.load()

loader = WebBaseLoader("https://lilianweng.github.io/posts/2024-02-05-human-data-quality/")
docs.extend(loader.load())


In [4]:
import uuid

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

chain = (
    {"doc": lambda x: x.page_content}
    | ChatPromptTemplate.from_template("Summarize the following document:\n\n{doc}")
    | ChatOpenAI(base_url="http://127.0.0.1:1234/v1/", max_retries=0)
    | StrOutputParser()
)

summaries = chain.batch(docs, {"max_concurrency": 5})

In [8]:
# from pprint import pprint 

pprint(summaries)

['This document provides an extensive overview of **LLM-powered autonomous '
 'agents**, defining them as systems where a Large Language Model (LLM) acts '
 'as the central "brain" to function as a powerful general problem solver, '
 'extending its capabilities beyond mere text generation.\n'
 '\n'
 'The core functionality of these sophisticated agents is built upon three '
 'essential pillars: Planning, Memory, and Tool Use.\n'
 '\n'
 '---\n'
 '\n'
 '### 🧠 I. Agent System Components\n'
 '\n'
 '**1. Planning (How Agents Think):**\n'
 'Planning techniques enable the agent to structure complex tasks into '
 'manageable steps and improve its reasoning over time.\n'
 '*   **Task Decomposition:** Methods like **Chain of Thought (CoT)** prompt '
 'the LLM to "think step by step," while **Tree of Thoughts (ToT)** explores '
 'multiple possible reasoning paths (creating a tree structure) at each '
 'stage.\n'
 '*   **Action-Observation Cycles:** **ReAct** integrates reasoning and action '
 'in

In [20]:
from langchain_core.stores import InMemoryByteStore
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
#vectorstore to use to index the child chunks
vectorstore = Chroma(collection_name="summaries",

    embedding_function= OpenAIEmbeddings(
        base_url="http://localhost:1234/v1", 
        api_key="lm-studio", 
        model="text-embedding-nomic-embed-text-v1.5",
        check_embedding_ctx_length=False # Explicitly set your local model name here
    )
)

# the storage layer for the parent documents
store = InMemoryByteStore()
id_key = "doc_id"

#the retriever
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key,
)

docs_ids = [str(uuid.uuid4()) for _ in docs]

#docs linked to summaries
summary_docs = [
    Document(page_content=s, metadata={id_key: docs_ids[i]})
    for i, s in enumerate(summaries)
]

#add
retriever.vectorstore.add_documents(summary_docs)
retriever.docstore.mset(list(zip(docs_ids,docs)))

In [22]:
query = "Memory in agents"
sub_docs = vectorstore.similarity_search(query, k=1)
pprint(sub_docs[0])

Document(metadata={'doc_id': '4b9446d7-1ce6-4cd1-8f74-eddd5f662385'}, page_content='This document provides an extensive overview of **LLM-powered autonomous agents**, defining them as systems where a Large Language Model (LLM) acts as the central "brain" to function as a powerful general problem solver, extending its capabilities beyond mere text generation.\n\nThe core functionality of these sophisticated agents is built upon three essential pillars: Planning, Memory, and Tool Use.\n\n---\n\n### 🧠 I. Agent System Components\n\n**1. Planning (How Agents Think):**\nPlanning techniques enable the agent to structure complex tasks into manageable steps and improve its reasoning over time.\n*   **Task Decomposition:** Methods like **Chain of Thought (CoT)** prompt the LLM to "think step by step," while **Tree of Thoughts (ToT)** explores multiple possible reasoning paths (creating a tree structure) at each stage.\n*   **Action-Observation Cycles:** **ReAct** integrates reasoning and action 

In [26]:
retrieved_docs = retriever.invoke(query,n_results=1)
pprint(retrieved_docs[0].page_content[0:500])

('\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 "LLM Powered Autonomous Agents | Lil'Log\n"
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 "Lil'Log\n"
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '|\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 'Posts\n'
 '\n'
 '\n'
 '\n'
 '\n'
 'Archive\n'
 '\n'
 '\n'
 '\n'
 '\n'
 'Search\n'
 '\n'
 '\n'
 '\n'
 '\n'
 'Tags\n'
 '\n'
 '\n'
 '\n'
 '\n'
 'FAQ\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '      LLM Powered Autonomous Agents\n'
 '    \n'
 'Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian '
 'Weng\n'
 '\n'
 '\n'
 ' \n'
 '\n'
 '\n'
 'Table of Contents\n'
 '\n'
 '\n'
 '\n'
 'Agent System Overview\n'
 '\n'
 'Component One: Planning\n'
 '\n'
 'Task Decomposition\n'
 '\n'
 'Self-Reflection\n